# Week 6 Project: E-commerce Return Reason Classifier

Online dataset project aligned to Week 6 flow:
1. Data curation
2. Data preprocessing
3. Baselines and evaluation
4. Robust routing with confidence threshold
5. Fine-tune-ready JSONL export


In [ ]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

random.seed(42)
np.random.seed(42)


## Data Curation
Load an online dataset and keep a manageable English subset.


In [ ]:
try:
    ds = load_dataset('amazon_reviews_multi', 'en', split='train[:12000]', trust_remote_code=True)
    raw_df = ds.to_pandas()[['review_title', 'review_body', 'stars']].dropna().copy()
    raw_df['text'] = (raw_df['review_title'].fillna('') + '. ' + raw_df['review_body'].fillna('')).str.strip()
except Exception:
    ds = load_dataset('yelp_review_full', split='train[:12000]')
    raw_df = ds.to_pandas()[['text', 'label']].dropna().copy()
    raw_df['review_title'] = ''
    raw_df['review_body'] = raw_df['text']
    raw_df['stars'] = raw_df['label'].astype(int) + 1

raw_df = raw_df[raw_df['text'].str.len() > 30].reset_index(drop=True)
raw_df.head(3)


## Preprocessing + Labeling
Create practical return-reason labels from review text.


In [ ]:
LABELS = ['defective', 'wrong_item', 'size_or_fit', 'late_delivery', 'not_as_described', 'changed_mind']


def normalize_text(x: str) -> str:
    x = x.lower().strip()
    x = re.sub(r'\s+', ' ', x)
    return x


def assign_return_reason(text: str, stars: int) -> str | None:
    t = normalize_text(text)
    if any(k in t for k in ['broken', 'defect', 'stopped working', 'damaged', 'does not work', "didn't work"]):
        return 'defective'
    if any(k in t for k in ['wrong item', 'wrong product', 'received different', 'not what i ordered', 'different item']):
        return 'wrong_item'
    if any(k in t for k in ['too small', 'too big', 'does not fit', "didn't fit", 'size was off']):
        return 'size_or_fit'
    if any(k in t for k in ['arrived late', 'late delivery', 'delayed shipping', 'shipping took too long']):
        return 'late_delivery'
    if any(k in t for k in ['not as described', 'misleading', 'looked different', 'description was wrong']):
        return 'not_as_described'
    if stars <= 2 and any(k in t for k in ['returned', 'returning', 'refund', 'not worth it', 'changed my mind']):
        return 'changed_mind'
    return None


df = raw_df.copy()
df['label'] = [assign_return_reason(t, int(s)) for t, s in zip(df['text'], df['stars'])]
df = df[df['label'].notna()].copy().reset_index(drop=True)
df = df[df['label'].isin(LABELS)].copy().reset_index(drop=True)

df['label'].value_counts()


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print('\nClass distribution (train):')
print(train_df['label'].value_counts())


## Baselines + Evaluation
Compare majority baseline with a text classifier.


In [ ]:
X_train_text = train_df['text'].tolist()
X_val_text = val_df['text'].tolist()
X_test_text = test_df['text'].tolist()
y_train = train_df['label'].tolist()
y_val = val_df['label'].tolist()
y_test = test_df['label'].tolist()

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_train = vec.fit_transform(X_train_text)
X_val = vec.transform(X_val_text)
X_test = vec.transform(X_test_text)

majority = DummyClassifier(strategy='most_frequent')
majority.fit(X_train, y_train)
maj_preds = majority.predict(X_test)

logreg = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1)
logreg.fit(X_train, y_train)
log_preds = logreg.predict(X_test)

print('Majority baseline')
print('accuracy:', round(accuracy_score(y_test, maj_preds), 4), 'macro_f1:', round(f1_score(y_test, maj_preds, average='macro'), 4))
print('\nLogistic regression baseline')
print('accuracy:', round(accuracy_score(y_test, log_preds), 4), 'macro_f1:', round(f1_score(y_test, log_preds, average='macro'), 4))


In [ ]:
print(classification_report(y_test, log_preds, digits=4, zero_division=0))
cm = pd.DataFrame(confusion_matrix(y_test, log_preds, labels=LABELS), index=LABELS, columns=LABELS)
cm


## Confidence-Gated Routing
Route low-confidence predictions to `manual_review`.


In [ ]:
THRESHOLD = 0.55

probs = logreg.predict_proba(X_test)
classes = list(logreg.classes_)
max_idx = probs.argmax(axis=1)
max_scores = probs.max(axis=1)
preds = [classes[i] for i in max_idx]

gated_preds = [p if s >= THRESHOLD else 'manual_review' for p, s in zip(preds, max_scores)]
coverage = np.mean([g != 'manual_review' for g in gated_preds])

kept_true = [t for t, g in zip(y_test, gated_preds) if g != 'manual_review']
kept_pred = [g for g in gated_preds if g != 'manual_review']
kept_acc = accuracy_score(kept_true, kept_pred) if kept_true else 0.0
kept_f1 = f1_score(kept_true, kept_pred, average='macro') if kept_true else 0.0

print('coverage:', round(float(coverage), 4))
print('kept_accuracy:', round(float(kept_acc), 4))
print('kept_macro_f1:', round(float(kept_f1), 4))


## Fine-Tune-Ready JSONL
Export training/validation sets in chat JSONL format.


In [ ]:
SYSTEM_PROMPT = (
    'You are a return-reason classifier for e-commerce support. '
    'Return exactly one label from: defective, wrong_item, size_or_fit, late_delivery, not_as_described, changed_mind.'
)


def messages_for(row):
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label']},
        ]
    }


def write_jsonl(dataframe, path):
    with open(path, 'w') as f:
        for _, r in dataframe.iterrows():
            f.write(json.dumps(messages_for(r), ensure_ascii=False) + '\n')

out_dir = Path('jsonl')
out_dir.mkdir(exist_ok=True)
train_path = out_dir / 'returns_train.jsonl'
val_path = out_dir / 'returns_val.jsonl'

write_jsonl(train_df, train_path)
write_jsonl(val_df, val_path)

print('wrote:', train_path, 'rows=', len(train_df))
print('wrote:', val_path, 'rows=', len(val_df))


## Optional Demo
Single prediction helper for quick testing.


In [ ]:
def classify_return_reason(text: str, threshold: float = 0.55):
    x = vec.transform([text])
    prob = logreg.predict_proba(x)[0]
    idx = int(np.argmax(prob))
    label = logreg.classes_[idx]
    score = float(prob[idx])
    if score < threshold:
        return {'label': 'manual_review', 'confidence': round(score, 4)}
    return {'label': label, 'confidence': round(score, 4)}

demo_samples = [
    'The jacket arrived two weeks late and I requested a refund.',
    'I received a different item than what I ordered.',
    'These shoes are too small and did not fit at all.',
    'The blender stopped working after one day and smells burnt.',
    'The product looked nothing like the photos in the description.',
    'I changed my mind and want to return it even though it works.',
    'Package arrived on time and quality is great.'
]

for i, text in enumerate(demo_samples, 1):
    print(f'{i}. {text}')
    print(classify_return_reason(text, threshold=THRESHOLD))
    print('-' * 80)
